In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score,
    f1_score, confusion_matrix, classification_report
)

In [2]:
raw = pd.read_csv("heart.csv")

In [3]:
columns = ['age','sex','cp','trestbps','chol','fbs','restecg',
           'thalach','exang','oldpeak','slope','ca','thal','target']

In [4]:
data = pd.DataFrame(raw, columns=columns)

In [5]:
print("Step 1 - Data loaded")
print(f"Shape: {data.shape[0]} rows, {data.shape[1]} columns")

Step 1 - Data loaded
Shape: 303 rows, 14 columns


# Now we start with data cleaning

In [6]:
data = data[data['ca'] < 4]
data = data[data['thal'] > 0]
print(f"\nAfter cleaning: {len(data)} rows remaining")


After cleaning: 296 rows remaining


# Will now Split the data in 80-20 for train and test

In [7]:
X = data.drop('target', axis=1)
Y = data['target']

In [8]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 23)
print(f"\nStep 2 - Data Splitted")
print(f"Training rows: {len(X_train)}")
print(f"Testing rows: {len(X_test)}")


Step 2 - Data Splitted
Training rows: 236
Testing rows: 60


# Train the Model

In [9]:
model = LogisticRegression(max_iter=1000, random_state=23)
model.fit(X_train, Y_train)
print("\nStep 3 - Model trained successfully")


Step 3 - Model trained successfully


In [10]:
Y_pred  = model.predict(X_test) 

# Time to Evaluate the Performance

In [11]:
acc = accuracy_score(Y_test, Y_pred)
rec = recall_score(Y_test, Y_pred)
prec = precision_score(Y_test, Y_pred)
f1 = f1_score(Y_test, Y_pred)

print("\nStep 4 - Model predictions Result are as such: \n")
print(f"Accuracy : {acc*100:.1f}%")
print(f"Recall : {rec*100:.1f}%")
print(f"Precision : {prec*100:.1f}%")
print(f"F1-Score : {f1*100:.1f}%")


Step 4 - Model predictions Result are as such: 

Accuracy : 88.3%
Recall : 100.0%
Precision : 82.9%
F1-Score : 90.7%


In [12]:
print()
print(classification_report(Y_test, Y_pred, target_names=['No Disease','Has Disease']))


              precision    recall  f1-score   support

  No Disease       1.00      0.73      0.84        26
 Has Disease       0.83      1.00      0.91        34

    accuracy                           0.88        60
   macro avg       0.91      0.87      0.88        60
weighted avg       0.90      0.88      0.88        60



# Confusion Matrix + Metrics Chart

In [13]:
cm =  confusion_matrix(Y_test, Y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
fig.suptitle("Heart Disease Prediction -  Linear Regression", fontsize = 14)

sns.heatmap(cm, annot = True, fmt='d', cmap = 'Blues', xticklabels=["No Disease", "Has Disease"], yticklabels=["No Disease", "Has Disease"], ax=ax)

ax.set_title("Confusion Matrix")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")

plt.tight_layout()
plt.savefig("Confusion_mat.png", dpi=150, bbox_inches='tight')
plt.close()

# Each Weight Graph

In [14]:
coefficients = model.coef_[0]
feature_names = X.columns.tolist()

coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Coefficient': coefficients
}).sort_values('Coefficient', key=abs, ascending=False).reset_index(drop=True)

print(coef_df.to_string(index=False))

 Feature  Coefficient
     sex    -1.373055
      ca    -1.066981
    thal    -1.052766
      cp     0.710814
   exang    -0.676826
   slope     0.475664
 oldpeak    -0.374452
     fbs     0.328364
 restecg     0.175716
 thalach     0.031491
trestbps    -0.028242
     age     0.019549
    chol    -0.002321


In [15]:
fig2, ax2 = plt.subplots(figsize=(9, 5))

colors_feat = ['#C44E52' if c > 0 else '#4C72B0'
               for c in coef_df['Coefficient'].tolist()]

ax2.barh(coef_df['Feature'].tolist(), coef_df['Coefficient'].tolist(), color=colors_feat)
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title("Feature Importance\n(red = toward disease  |  blue = away from disease)")
ax2.set_xlabel("Coefficient value")
ax2.invert_yaxis()
plt.tight_layout()
plt.savefig("Feature_weight.png", dpi=150, bbox_inches='tight')
plt.close()

# Metrics Bar Chart of the Result

In [17]:
metrics = ['Accuracy', 'Recall', 'Precision', 'F1-Score']
scores  = [acc, rec, prec, f1]
colors  = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

fig3, ax3 = plt.subplots(figsize=(7, 5))
fig3.suptitle("Heart Disease Prediction — Logistic Regression", fontsize=13)

bars = ax3.bar(metrics, [s * 100 for s in scores], color=colors, width=0.5)

ax3.set_ylim(0, 115)
ax3.set_title("Model Performance Summary")
ax3.set_ylabel("Score (%)")
ax3.set_xlabel("Metric")

for bar, score in zip(bars, scores):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 1.5,
        f"{score * 100:.1f}%",
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

plt.tight_layout()
plt.savefig("performance_summary.png", dpi=150, bbox_inches='tight')
plt.close()
print("Performance summary chart saved.")

Performance summary chart saved.
